# Evaluate DP Synthetic Data

Runs the same four evaluators used in `03_evaluation_debug.ipynb` (Statistical, TSTR, MIA, Distance Metrics) over the **18 DP synthetic CSVs** (3 generators × 3 epsilons × 2 datasets currently available: `diabetes_130us`, `acs_income`). `home_credit` is left in `DATASETS` but will simply raise `FileNotFoundError` per missing file and be skipped/logged — once those 9 files land from Kaggle, just rerun this notebook with no code changes.

Output:
- Combines with the existing `outputs/results/baseline_scores.csv` (9 non-DP rows)
- Writes `outputs/results/full_benchmark.csv` (all rows, wide format)
- Writes `outputs/results/utility_scores.csv` and `outputs/results/privacy_scores.csv` (split, per `config.yaml` → `reporting`) for the Stage 6 aggregator

In [1]:
import os
os.chdir('..')

import json
import yaml
import pandas as pd
import numpy as np
from pathlib import Path

with open('config.yaml') as f:
    config = yaml.safe_load(f)

from src.evaluation.utility.statistical import StatisticalEvaluator
from src.evaluation.utility.tstr import TSTREvaluator
from src.evaluation.privacy.mia import MIAEvaluator
from src.evaluation.privacy.distance_metrics import DistanceMetricsEvaluator

stat_eval  = StatisticalEvaluator(config)
tstr_eval  = TSTREvaluator(config)
mia_eval   = MIAEvaluator(config)
dist_eval  = DistanceMetricsEvaluator(config)

# home_credit stays in the list on purpose — missing files are caught and
# logged per-combo below, so rerunning this notebook later (once the Kaggle
# run finishes) fills it in with zero code changes.
DATASETS   = ['diabetes_130us', 'acs_income', 'home_credit']
GENERATORS = config['generators']['list']                     # ['ctgan', 'tvae', 'copulagan']
EPSILONS   = config['differential_privacy']['epsilons']       # [1, 5, 10]

print(f'Evaluators loaded. Epsilons from config: {EPSILONS}')

Evaluators loaded. Epsilons from config: [1, 5, 10]


In [2]:
def load_dataset(config, dataset_name):
    """Load real train/test CSVs and restore categorical dtypes from meta.json."""
    processed_dir = Path(config['datasets'][dataset_name]['processed_dir'])
    train_df = pd.read_csv(processed_dir / 'train.csv')
    test_df  = pd.read_csv(processed_dir / 'test.csv')

    with open(processed_dir / 'meta.json') as f:
        meta = json.load(f)

    for col, col_meta in meta.get('columns', {}).items():
        if col_meta['type'] == 'categorical':
            if col in train_df.columns:
                train_df[col] = train_df[col].astype(str)
                test_df[col]  = test_df[col].astype(str)

    return train_df, test_df, meta


def load_synthetic(config, dataset_name, label):
    """Load a synthetic CSV. Raises FileNotFoundError if not yet generated."""
    synth_path = Path(config['datasets'][dataset_name]['synthetic_dir']) / f'{label}.csv'
    if not synth_path.exists():
        raise FileNotFoundError(f'Synthetic file not found: {synth_path}')
    return pd.read_csv(synth_path)


def evaluate_combo(dataset_name, target_col, real_train, real_test, gen_name, epsilon_label, label):
    """Run all 4 evaluators for one (dataset, generator, epsilon) combo and return a flat row dict."""
    synthetic = load_synthetic(config, dataset_name, label)

    # Align synthetic dtypes with real (same pattern as baseline notebook)
    for col in synthetic.columns:
        if col in real_train.columns:
            if real_train[col].dtype == object:
                synthetic[col] = synthetic[col].astype(str)

    stat_res = stat_eval.evaluate(real_train, synthetic, dataset_name, label)
    tstr_res = tstr_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)
    mia_res  = mia_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)
    dist_res = dist_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)

    return {
        'dataset':            dataset_name,
        'generator':          gen_name,
        'epsilon':            epsilon_label,
        'label':              label,
        # Utility
        'stat_overall':       stat_res['overall_score'],
        'wasserstein_mean':   stat_res['wasserstein_mean'],
        'corr_diff':          stat_res['correlation_matrix_diff'],
        'cat_similarity':     stat_res['categorical_similarity_mean'],
        'tstr_auc':           tstr_res['tstr']['roc_auc'],
        'trtr_auc':           tstr_res['trtr']['roc_auc'],
        'tstr_f1':            tstr_res['tstr']['f1_weighted'],
        'tstr_accuracy':      tstr_res['tstr']['accuracy'],
        'utility_ratio':      tstr_res['utility_ratio'],
        # Privacy
        'mia_auc':            mia_res['mia_auc'],
        'mia_privacy_score':  mia_res['privacy_score'],
        'dcr_mean':           dist_res['dcr_mean'],
        'dcr_median':         dist_res['dcr_median'],
        'nndr_mean':          dist_res['nndr_mean'],
        'nndr_median':        dist_res['nndr_median'],
        'dist_privacy_score': dist_res['privacy_score'],
    }


print('Helper functions defined.')

Helper functions defined.


In [3]:
dp_results = []
skipped = []

for dataset_name in DATASETS:
    print(f'\n{"="*60}')
    print(f'  Dataset: {dataset_name}')
    print(f'{"="*60}')

    target_col = config['datasets'][dataset_name]['target_col']

    try:
        real_train, real_test, meta = load_dataset(config, dataset_name)
    except FileNotFoundError as e:
        print(f'  SKIPPING dataset entirely — processed data not found: {e}')
        skipped.append((dataset_name, 'ALL', 'ALL'))
        continue

    for gen_name in GENERATORS:
        for eps in EPSILONS:
            label = f'{gen_name}_eps{eps}'
            print(f'\n  [{label}]')

            try:
                row = evaluate_combo(dataset_name, target_col, real_train, real_test, gen_name, eps, label)
                dp_results.append(row)
                print(f'    TSTR AUC: {row["tstr_auc"]:.4f} | MIA AUC: {row["mia_auc"]:.4f} | DCR: {row["dcr_mean"]:.4f}')

            except FileNotFoundError as e:
                print(f'    SKIPPED (not generated yet): {e}')
                skipped.append((dataset_name, label, str(e)))

            except Exception as e:
                print(f'    ERROR: {e}')
                skipped.append((dataset_name, label, f'ERROR: {e}'))

print(f'\n\nDP evaluation complete. {len(dp_results)} combos scored, {len(skipped)} skipped.')
if skipped:
    print('\nSkipped combos (expected for home_credit until Kaggle run finishes):')
    for s in skipped:
        print(f'  {s[0]} / {s[1]}')


  Dataset: diabetes_130us

  [ctgan_eps1]
    TSTR AUC: 0.5354 | MIA AUC: 0.6342 | DCR: 7.3032

  [ctgan_eps5]
    TSTR AUC: 0.5633 | MIA AUC: 0.6230 | DCR: 8.5538

  [ctgan_eps10]
    TSTR AUC: 0.5914 | MIA AUC: 0.6298 | DCR: 7.4561

  [tvae_eps1]
    TSTR AUC: 0.6158 | MIA AUC: 0.6452 | DCR: 5.9422

  [tvae_eps5]
    TSTR AUC: 0.6059 | MIA AUC: 0.6493 | DCR: 6.0210

  [tvae_eps10]
    TSTR AUC: 0.6106 | MIA AUC: 0.6799 | DCR: 5.8751

  [copulagan_eps1]
    TSTR AUC: 0.5441 | MIA AUC: 0.6969 | DCR: 49.6982

  [copulagan_eps5]
    TSTR AUC: 0.5413 | MIA AUC: 0.7032 | DCR: 49.1171

  [copulagan_eps10]
    TSTR AUC: 0.5248 | MIA AUC: 0.7049 | DCR: 50.0871

  Dataset: acs_income

  [ctgan_eps1]
    TSTR AUC: 0.8644 | MIA AUC: 0.6267 | DCR: 23.4972

  [ctgan_eps5]
    TSTR AUC: 0.8598 | MIA AUC: 0.6247 | DCR: 23.4031

  [ctgan_eps10]
    TSTR AUC: 0.8591 | MIA AUC: 0.6116 | DCR: 25.9075

  [tvae_eps1]
    TSTR AUC: 0.8786 | MIA AUC: 0.6401 | DCR: 7.7562

  [tvae_eps5]
    TSTR AUC: 0.8836

## Merge with baseline and write Stage 6-ready outputs

- `full_benchmark.csv`: every row (baseline + DP), wide format — exactly what `reporting/aggregator.py` should read from.
- `utility_scores.csv` / `privacy_scores.csv`: same rows split into the two files named in `config['reporting']`, in case the aggregator is written to expect them separately.

In [4]:
dp_df = pd.DataFrame(dp_results)

baseline_path = Path('outputs/results/baseline_scores.csv')
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    full_df = pd.concat([baseline_df, dp_df], ignore_index=True)
    print(f'Merged {len(baseline_df)} baseline rows + {len(dp_df)} DP rows = {len(full_df)} total rows.')
else:
    full_df = dp_df
    print(f'WARNING: {baseline_path} not found — writing DP-only rows ({len(dp_df)}).')

# Dedup safety net: if this notebook is rerun after home_credit lands,
# keep the latest row per (dataset, label) instead of duplicating.
full_df = full_df.drop_duplicates(subset=['dataset', 'label'], keep='last').reset_index(drop=True)

utility_cols = ['dataset', 'generator', 'epsilon', 'label',
                 'stat_overall', 'wasserstein_mean', 'corr_diff', 'cat_similarity',
                 'tstr_auc', 'trtr_auc', 'tstr_f1', 'tstr_accuracy', 'utility_ratio']
privacy_cols = ['dataset', 'generator', 'epsilon', 'label',
                 'mia_auc', 'mia_privacy_score',
                 'dcr_mean', 'dcr_median', 'nndr_mean', 'nndr_median', 'dist_privacy_score']

reporting_cfg = config['reporting']
Path(reporting_cfg['output_file']).parent.mkdir(parents=True, exist_ok=True)

full_df.to_csv(reporting_cfg['output_file'], index=False)
full_df[utility_cols].to_csv(reporting_cfg['utility_file'], index=False)
full_df[privacy_cols].to_csv(reporting_cfg['privacy_file'], index=False)

print(f"\nSaved:")
print(f"  {reporting_cfg['output_file']}  ({len(full_df)} rows)")
print(f"  {reporting_cfg['utility_file']}")
print(f"  {reporting_cfg['privacy_file']}")

print(f"\nRows per dataset:")
print(full_df.groupby('dataset').size())
print(f"\nRows per (dataset, epsilon):")
print(full_df.groupby(['dataset', 'epsilon']).size())

Merged 9 baseline rows + 18 DP rows = 27 total rows.

Saved:
  outputs/results/full_benchmark.csv  (27 rows)
  outputs/results/utility_scores.csv
  outputs/results/privacy_scores.csv

Rows per dataset:
dataset
acs_income        12
diabetes_130us    12
home_credit        3
dtype: int64

Rows per (dataset, epsilon):
dataset         epsilon
acs_income      1          3
                5          3
                10         3
                nodp       3
diabetes_130us  1          3
                5          3
                10         3
                nodp       3
home_credit     nodp       3
dtype: int64


In [5]:
# Quick sanity check: utility should trend down and MIA AUC should trend
# toward 0.5 as epsilon decreases (Stage 5 checkpoint from project_context.md).
# 'nodp' sorts after numeric epsilons here, which is fine for eyeballing.
pivot = full_df.pivot_table(
    index=['dataset', 'generator'],
    columns='epsilon',
    values=['utility_ratio', 'mia_auc'],
)
pivot

mia_auc                                \
epsilon                          1         5        10      nodp   
dataset        generator                                           
acs_income     copulagan  0.700446  0.697820  0.690635  0.604143   
               ctgan      0.626707  0.624719  0.611591  0.637356   
               tvae       0.640147  0.631966  0.654655  0.689626   
diabetes_130us copulagan  0.696862  0.703246  0.704946  0.615658   
               ctgan      0.634181  0.623014  0.629779  0.636841   
               tvae       0.645153  0.649272  0.679936  0.664701   
home_credit    copulagan       NaN       NaN       NaN  0.695781   
               ctgan           NaN       NaN       NaN  0.689894   
               tvae            NaN       NaN       NaN  0.689539   

                         utility_ratio                                
epsilon                              1         5        10      nodp  
dataset        generator                                              
acs_income     copulagan      0.942536  0.760188  0.873975  0.972927  
               ctgan          0.965463  0.960250  0.959520  0.974668  
               tvae           0.981315  0.987077  0.974639  0.961008  
diabetes_130us copulagan      0.796478  0.792414  0.768190  0.805716  
               ctgan          0.783746  0.824586  0.865758  0.843707  
               tvae           0.900032  0.885013  0.892119  0.903520  
home_credit    copulagan           NaN       NaN       NaN  0.921013  
               ctgan               NaN       NaN       NaN  0.922612  
               tvae                NaN       NaN       NaN  0.817494